# Inspect simulated scan data (`SimulateScanPlugin`)

Phase-1 simulation: a deterministic, unity-gain antenna-temperature TOD (Kelvin) that
replaces `scan_data.visibility`. Components: beam-convolved Galactic foreground (Simeer) +
atmospheric + spillover + receiver + noise-diode temperatures.

This notebook loads the stored context and checks the result is physically sensible.

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from museek.enums.result_enum import ResultEnum

plt.rcParams["figure.figsize"] = [12, 5]
plt.rcParams["font.size"] = 11

# --- parameters ---
context_folder = "/home/mgrsantos/projects/data/context"
block_name = "1675021905"
pickle_name = "simulate_scan_plugin.pickle"

pickle_path = Path(context_folder) / block_name / pickle_name
print("Loading:", pickle_path, "| exists:", pickle_path.exists())

In [ ]:
with open(pickle_path, "rb") as f:
    ctx = pickle.load(f)
scan_data = ctx.get(ResultEnum.SCAN_DATA).result

vis = np.asarray(scan_data.visibility.array).real.astype(np.float32)   # (n_time, n_freq, n_recv)
freq_MHz = scan_data.frequencies.squeeze / 1e6
ts = scan_data.timestamps.squeeze
t_min = (ts - ts[0]) / 60.0
receivers = [r.name for r in scan_data.receivers]

# Recover antenna temperature in K by dividing out the deterministic gain the simulation applied.
# (1) read calibrator gain (ReadCalibratorGainsPlugin -> gain_solution, counts/K, per freq & recv);
# (2) synthetic gain (smooth poly x standing waves, a function of frequency only, stored by
#     SimulateScanPlugin). With both removed, vis = T_total x (1 + 1/f) x (1 + white); a time-median
#     over many dumps averages the zero-mean noise away, recovering Tsys. Zero-gain channels
#     (band edges / RFI) have vis == 0 already and are flagged, so we leave them at 0 (not NaN).
# Multiply in place to keep vis float32 (no float64 copy).
_rg = ctx.get(ResultEnum.CALIBRATOR_GAIN)
read_gain = None if _rg is None else np.asarray(_rg.result, dtype=np.float32)   # (n_freq, n_recv)
if read_gain is not None:
    with np.errstate(divide="ignore", invalid="ignore"):
        inv_gain = np.where(read_gain > 0, 1.0 / read_gain, 0.0).astype(np.float32)
    vis *= inv_gain[np.newaxis]                                      # in-place, stays float32
    gmed = np.nanmedian(read_gain[read_gain > 0])
    print(f"Divided out read calibrator gain (median {gmed:.1f} counts/K)")
else:
    print("No read calibrator gain present")

_sg = ctx.get(ResultEnum.SIMULATED_SYNTH_GAIN)
synth_gain = None if _sg is None else _sg.result
if synth_gain is not None:
    synth_gain = np.asarray(synth_gain, dtype=np.float32)            # (n_freq,)
    vis /= synth_gain[np.newaxis, :, np.newaxis]
    print(f"Divided out synthetic gain (range {synth_gain.min():.4f}-{synth_gain.max():.4f}) -> vis in K")
else:
    print("No synthetic gain stored -> vis in K (read gain only removed)")

flag_names = scan_data.flags.flag_names

# noise-diode-on mask (per dump)
nd_on = np.zeros(vis.shape[0], dtype=bool)
if "noise_diode_on" in flag_names:
    nd_on = np.asarray(
        scan_data.flags.get(freq=0, recv=0).array[flag_names.index("noise_diode_on")]
    ).squeeze().astype(bool)

# "bad data" mask for plotting: OR of every flag EXCEPT the noise diode (which is real signal,
# not a defect). True = flagged. `vism` is the masked visibility; use it in the plots below so
# flagged channels/times are blanked instead of showing up as zeros.
bad_mask = np.zeros(vis.shape, dtype=bool)               # (n_time, n_freq, n_recv)
for idx, name in enumerate(flag_names):
    if name == "noise_diode_on":
        continue
    bad_mask |= np.asarray(scan_data.flags._flags[idx].array).astype(bool)
vism = np.ma.masked_array(vis, mask=bad_mask)

# Free the heavy arrays we no longer need so the 3 GB pickle does not stay pinned for the whole
# notebook: the original visibility (now in `vis`), the per-dump `weights` (unused here) and the
# full time x freq x recv `gain_solution` (only its [0] slice -> `read_gain` was needed). We keep
# scan_data.flags and the coordinate/time arrays, which later cells still use. `del ctx` drops the
# context wrapper and any other stored results. ~2.4 GB freed.
scan_data.visibility = None
scan_data.weights = None
del ctx
import gc; gc.collect()

print("vis shape:", vis.shape, "| receivers:", receivers)
print("frequencies:", f"{freq_MHz.min():.1f} - {freq_MHz.max():.1f} MHz ({len(freq_MHz)} ch)")
print("duration:", f"{t_min[-1]:.1f} min ({len(t_min)} dumps)")
print(f"flagged (non-diode) fraction: {bad_mask.mean()*100:.1f}%")

## 1. Sanity-check summary

In [ ]:
band = (freq_MHz > 600) & (freq_MHz < 950)   # clean mid-band for stats

print(f"flagged (non-diode) fraction: {vism.mask.mean()*100:.1f}%")
print(f"global range: [{vism.min():.2f}, {vism.max():.2f}] K, median {np.ma.median(vism):.2f} K")
print()
if nd_on.any():
    on_T = vism[nd_on][:, band, 0].mean()
    off_T = vism[~nd_on][:, band, 0].mean()
    print(f"noise diode: {int(nd_on.sum())}/{len(nd_on)} on-dumps | "
          f"T(on)={on_T:.2f} K  T(off)={off_T:.2f} K  excess={on_T - off_T:.2f} K")
print()
print(f"{'receiver':>8} | {'mid-band median [K]':>20} | {'off-dump baseline [K]':>22}")
for i, name in enumerate(receivers):
    base = np.ma.median(vism[~nd_on][:, band, i]) if nd_on.any() else np.ma.median(vism[:, band, i])
    print(f"{name:>8} | {np.ma.median(vism[:, band, i]):>20.2f} | {base:>22.2f}")

# median over receivers vs frequency (noise-diode OFF dumps), in ~50 MHz bins
tsys_fr = np.ma.median(vism[~nd_on], axis=0)          # (n_freq, n_recv) off-dump time-median
med_recv = np.ma.median(tsys_fr, axis=1)              # (n_freq,) median across receivers
print()
print(f"{'freq [MHz]':>10} | {'median over receivers [K]':>26}")
lo0 = 50 * np.floor(freq_MHz.min() / 50)
for lo in np.arange(lo0, freq_MHz.max() + 50, 50):
    inb = (freq_MHz >= lo) & (freq_MHz < lo + 50)
    if inb.any():
        val = np.ma.median(med_recv[inb])             # median over channels in the 50 MHz bin
        tag = " (masked)" if np.ma.is_masked(val) else ""
        print(f"{lo + 25:>10.0f} | {('' if np.ma.is_masked(val) else f'{val:.2f}'):>26}{tag}")

## 1b. System temperature vs frequency (HH / VV, all dishes)

Time-median of $T_{\rm sys}$ per channel, flags applied, split by polarisation. Cell 2 divides out
**both** the read calibrator gain and the synthetic standing-wave gain, so this is the underlying
antenna temperature in Kelvin with no gain applied (the 1/f and white noise are zero-mean and average
out under the time median). Noise-diode-ON dumps are excluded, so this is the off-source $T_{\rm sys}$.

In [ ]:
# time-median Tsys per channel (ND-off, flags applied via vism), gain fully removed -> Kelvin
tsys_off = np.ma.median(vism[~nd_on], axis=0)        # (n_freq, n_recv), masked
pol_of = [name[-1].lower() for name in receivers]    # 'h' or 'v' per receiver

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, pol, label in zip(axes, ("h", "v"), ("HH", "VV")):
    for i, name in enumerate(receivers):
        if pol_of[i] != pol:
            continue
        ax.plot(freq_MHz, np.ma.filled(tsys_off[:, i], np.nan), lw=0.7, label=name[:-1])
    ax.set_xlabel("Frequency [MHz]")
    ax.set_title(label)
    ax.grid(alpha=0.3)
    ax.legend(ncol=2, fontsize=8)
axes[0].set_ylabel(r"time-median $T_{\rm sys}$ [K]  (ND off, flags applied)")
fig.suptitle("System temperature vs frequency (no gain applied) \u2014 per dish, HH and VV")
plt.tight_layout()
plt.show()

## 1c. Full applied gain vs frequency (HH / VV, all dishes)

The total deterministic gain the simulation multiplied onto the antenna temperature,
`gain = read_calibrator_gain(f, recv) x synth_gain(f)`, in counts/K, split by polarisation
(one line per dish). The read gain is per dish and per channel; the synthetic gain (smooth
polynomial x standing waves) is the same function of frequency for every dish, so the ~1%
standing-wave ripple is common to all curves. This is exactly what cell 2 divides out to put
the plots in Kelvin. Zero-gain channels (band edges / RFI) are blanked. The 1/f and white
noise are not shown here -- they are not part of the deterministic gain.

In [ ]:
# full deterministic gain per receiver = read calibrator gain (counts/K) x synthetic gain (dimensionless)
if read_gain is None and synth_gain is None:
    print("No gain was applied (read_gain and synth_gain both absent) -> gain is unity; nothing to plot.")
else:
    n_recv = vis.shape[2]
    full_gain = np.ones((freq_MHz.size, n_recv), dtype=float)
    if read_gain is not None:
        full_gain = full_gain * read_gain
    if synth_gain is not None:
        full_gain = full_gain * synth_gain[:, np.newaxis]
    full_gain = np.where(full_gain > 0, full_gain, np.nan)   # blank zero-gain (band-edge / RFI) channels

    pol_of = [name[-1].lower() for name in receivers]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for ax, pol, label in zip(axes, ("h", "v"), ("HH", "VV")):
        for i, name in enumerate(receivers):
            if pol_of[i] != pol:
                continue
            ax.plot(freq_MHz, full_gain[:, i], lw=0.7, label=name[:-1])
        ax.set_xlabel("Frequency [MHz]")
        ax.set_title(label)
        ax.grid(alpha=0.3)
        ax.legend(ncol=2, fontsize=8)
    axes[0].set_ylabel("full applied gain [counts/K]")
    fig.suptitle("Full deterministic gain (read calibrator x synthetic) \u2014 per dish, HH and VV")
    plt.tight_layout()
    plt.show()

## 2. Waterfall (time × frequency)

Expect a smooth beam-convolved foreground drift, RFI-free (this is a pure simulation), with
the band edges sitting at the receiver-model extrapolation level and periodic noise-diode stripes.

In [ ]:
i_recv = 0   # index into `receivers`

cmap = plt.cm.viridis.copy(); cmap.set_bad("0.6")   # flagged pixels in grey
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(vism[:, :, i_recv].T, aspect="auto", origin="lower",
               extent=[t_min[0], t_min[-1], freq_MHz.min(), freq_MHz.max()], cmap=cmap)
ax.set_xlabel("Time [min]")
ax.set_ylabel("Frequency [MHz]")
ax.set_title(f"Simulated $T_{{\\rm ant}}$ waterfall \u2014 {receivers[i_recv]} (flags masked)")
plt.colorbar(im, label="T [K]")
plt.tight_layout()
plt.show()

## 2b. Waterfall — noise-diode OFF dumps only

Same as above but excluding the `noise_diode_on` dumps, so the periodic diode stripes are
removed and the underlying foreground / system-temperature drift is easier to see.
(Columns are the off-dumps in time order; the brief ND-on gaps are dropped.)

In [ ]:
i_recv = 0
vis_off = vism[~nd_on, :, i_recv]          # (n_off, n_freq), masked
time_off = t_min[~nd_on]

cmap = plt.cm.viridis.copy(); cmap.set_bad("0.6")
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(vis_off.T, aspect="auto", origin="lower",
               extent=[time_off[0], time_off[-1], freq_MHz.min(), freq_MHz.max()], cmap=cmap)
ax.set_xlabel("Time [min]")
ax.set_ylabel("Frequency [MHz]")
ax.set_title(f"Simulated $T_{{\\rm ant}}$ \u2014 {receivers[i_recv]} (noise-diode OFF dumps, flags masked)")
plt.colorbar(im, label="T [K]")
plt.tight_layout()
plt.show()

## 2c. Residual waterfall — per-frequency time median subtracted, flags masked (ND off)

Diode-off dumps only, with each frequency channel's time-**median** (over unflagged samples)
subtracted and the combined flags applied (masked pixels in grey). Removing the median (rather
than the mean) is robust to outlier dumps and leaves a residual whose per-channel time median is
zero by construction (verified in 2d).

In [ ]:
i_recv = 0
off = ~nd_on

# combined flags for this receiver (band-edge/known RFI, rawdata, elevation/outlier, ...)
flags = np.asarray(scan_data.flags.combine(threshold=1).get(recv=i_recv).squeeze).astype(bool)  # (n_time, n_freq)

vis_off = np.ma.masked_array(vis[off, :, i_recv], mask=flags[off])   # (n_off, n_freq), masked
time_med = np.ma.median(vis_off, axis=0)                             # per-freq time median over UNflagged samples
resid = vis_off - time_med[np.newaxis, :]
time_off = t_min[off]

vlim = np.percentile(np.abs(resid.compressed()), 99)                 # robust symmetric scale (unmasked only)
cmap = plt.cm.RdBu_r.copy()
cmap.set_bad("0.6")                                                  # masked pixels in grey

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(resid.T, aspect="auto", origin="lower",
               extent=[time_off[0], time_off[-1], freq_MHz.min(), freq_MHz.max()],
               cmap=cmap, vmin=-vlim, vmax=vlim)
ax.set_xlabel("Time [min]")
ax.set_ylabel("Frequency [MHz]")
ax.set_title(f"$T_{{\\rm ant}}$ residual (per-freq time median removed), flags masked \u2014 {receivers[i_recv]}, ND off")
plt.colorbar(im, label="$\\Delta T$ [K]")
plt.tight_layout()
plt.show()

print(f"masked fraction (ND-off): {resid.mask.mean()*100:.1f}%")

## 2c-2. Residual spectra at a few times (median removed, ND off)

Slices of the 2c residual (per-frequency time median removed, flags masked) at a few diode-off
times — the spectral shape of the time-varying signal at those dumps, with the static bandpass
already subtracted. Masked (flagged) channels appear as gaps.

In [ ]:
# uses `resid` and `time_off` from 2c (same i_recv)
n_off = resid.shape[0]
sel = [0, n_off // 4, n_off // 2, 3 * n_off // 4, n_off - 1]

fig, ax = plt.subplots(figsize=(12, 5))
for k in sel:
    ax.plot(freq_MHz, np.ma.filled(resid[k], np.nan), lw=0.7, label=f"t={time_off[k]:.0f} min")
ax.axhline(0.0, color="k", lw=0.5)
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel(r"$\Delta T$ [K]  (median removed)")
ax.set_title(f"Residual spectra at selected times \u2014 {receivers[i_recv]}, ND off")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2c-3. Residual spectra minus a smooth polynomial (time + frequency averaged)

Per selected time we average the residual over `n_avg_time` dumps and boxcar-smooth over
`n_avg_freq` channels before removing a degree-`deg` polynomial. `vis` here still contains the
1/f + white noise (only the deterministic gains are divided out), so a single dump is
noise-dominated; averaging suppresses the zero-mean noise and reveals the slowly-varying,
broad beam-modulation structure from the foreground×beam convolution. (Both averaging widths
are tunable.)

In [ ]:
deg = 9            # polynomial degree for the smooth bandpass/foreground part
n_avg_time = 31    # dumps averaged around each selected time (suppresses 1/f + white noise)
n_avg_freq = 5     # channels boxcar-smoothed (the beam-modulation waves are broader than 1 channel)

def _smooth_freq(masked_row, n):
    """Boxcar-average a masked (n_freq,) row over n channels, ignoring masked/NaN."""
    a = np.ma.filled(np.ma.asarray(masked_row, dtype=float), np.nan)
    ok = np.isfinite(a).astype(float)
    kern = np.ones(n)
    num = np.convolve(np.where(np.isfinite(a), a, 0.0), kern, mode="same")
    den = np.convolve(ok, kern, mode="same")
    with np.errstate(invalid="ignore", divide="ignore"):
        out = num / den
    out[den == 0] = np.nan
    return out

def _detrended_smoothed(resid_arr, k):
    """Time-average resid_arr over +/- n_avg_time/2 dumps around k, remove a degree-deg poly, smooth in freq."""
    w = n_avg_time // 2
    lo, hi = max(0, k - w), min(resid_arr.shape[0], k + w + 1)
    row = np.ma.mean(resid_arr[lo:hi], axis=0)                       # (n_freq,) masked, time-averaged
    good = ~np.ma.getmaskarray(row) & np.isfinite(row.data)
    p = np.polynomial.Polynomial.fit(freq_MHz[good], row.data[good], deg)
    diff = np.ma.masked_array(row.data - p(freq_MHz), mask=~good)
    return _smooth_freq(diff, n_avg_freq)

fig, ax = plt.subplots(figsize=(12, 5))
diffs = []
for k in sel:                                  # `sel`, `resid`, `time_off` from 2c / 2c-2
    d = _detrended_smoothed(resid, k)
    ax.plot(freq_MHz, d, lw=0.9, label=f"t\u2248{time_off[k]:.0f} min")
    diffs.append(d)

# robust symmetric y-limit from features AWAY from the 600 MHz spike (598-602 MHz excluded)
stack = np.array(diffs)
away = stack[:, (freq_MHz < 598) | (freq_MHz > 602)]
lim = np.nanpercentile(np.abs(away), 99.5)
ax.set_ylim(-1.3 * lim, 1.3 * lim)

ax.axhline(0.0, color="k", lw=0.5)
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel(r"residual $-$ poly fit [K]")
ax.set_title(f"Residual minus degree-{deg} poly, {n_avg_time}-dump & {n_avg_freq}-ch averaged \u2014 {receivers[i_recv]}, ND off")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2c-4. Residual spectra minus polynomial — cross-pol companion (time + frequency averaged)

Same as 2c-3 for the cross-polarisation companion of `i_recv`, reusing the same averaging helpers.

In [ ]:
# cross-pol companion of whatever i_recv is set to in cell 2c; reuses the helpers from 2c-3
recv_name = receivers[i_recv]
companion = recv_name[:-1] + ("v" if recv_name.endswith("h") else "h")
i2 = receivers.index(companion)
off = ~nd_on
fl2 = np.asarray(scan_data.flags.combine(threshold=1).get(recv=i2).squeeze).astype(bool)
v2 = np.ma.masked_array(vis[off, :, i2], mask=fl2[off])
resid2 = v2 - np.ma.median(v2, axis=0)[None, :]
time_off = t_min[off]
n_off2 = resid2.shape[0]
sel2 = [0, n_off2 // 4, n_off2 // 2, 3 * n_off2 // 4, n_off2 - 1]

fig, ax = plt.subplots(figsize=(12, 5))
diffs = []
for k in sel2:
    d = _detrended_smoothed(resid2, k)          # same time+freq averaging as 2c-3
    ax.plot(freq_MHz, d, lw=0.9, label=f"t\u2248{time_off[k]:.0f} min")
    diffs.append(d)

stack = np.array(diffs)
away = stack[:, (freq_MHz < 598) | (freq_MHz > 602)]
lim = np.nanpercentile(np.abs(away), 99.5)
ax.set_ylim(-1.3 * lim, 1.3 * lim)

ax.axhline(0.0, color="k", lw=0.5)
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel(r"residual $-$ poly fit [K]")
ax.set_title(f"Residual minus degree-{deg} poly, {n_avg_time}-dump & {n_avg_freq}-ch averaged \u2014 {receivers[i2]}, ND off")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2d. RMS over time per frequency — all receivers

For each receiver: RMS over time (diode-off, unflagged) of the per-frequency median-subtracted
residual, as a spectrum. Measures how much each channel varies in time — the amplitude of the
pointing-driven foreground drift per frequency, with the static bandpass removed.

In [ ]:
off = ~nd_on
all_flags = scan_data.flags.combine(threshold=1)

fig, ax = plt.subplots(figsize=(12, 4))
for i, name in enumerate(receivers):
    fl = np.asarray(all_flags.get(recv=i).squeeze).astype(bool)         # (n_time, n_freq)
    v = np.ma.masked_array(vis[off, :, i], mask=fl[off])
    r = v - np.ma.median(v, axis=0)[np.newaxis, :]                      # per-freq time median removed
    rms_time = np.ma.sqrt(np.ma.mean(r**2, axis=0))                     # (n_freq,) RMS over time
    ax.plot(freq_MHz, np.ma.filled(rms_time, np.nan), lw=0.7, label=name)
    print(f"{name}: median time-RMS across band = {np.ma.median(rms_time):.3f} K")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel(r"RMS over time [K]")
ax.set_title("Time RMS per frequency of residual \u2014 all receivers, ND off")
ax.legend(ncol=3, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2e. Sky map (RA/Dec) of the 2c residual — single channel (700 MHz), smoothed

The diode-off pointings (median az/el + timestamp \u2192 RA/Dec) are gridded and Gaussian-smoothed
(0.5\u00b0) to look like a map. Coverage is non-uniform (a thin scan track), so we smooth the
value-sum and the hit-count separately and divide \u2014 the smoothed map is the coverage-weighted
mean $\Delta T$ at 700 MHz, blanked where coverage is too thin. (RA smoothing ignores the small
cos(dec) factor over this narrow dec range.)

In [ ]:
import astropy.units as u
import scipy.ndimage as ndi
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.time import Time

from museek.plugin.point_source_calibration_plugin import (
    calculate_median_coordinates_excluding_flagged_antennas,
)

i_recv = 0
freq_target_MHz = 700.0
pix_deg = 0.1            # map pixel size
smoothing_deg = 0.3      # Gaussian sigma
cover_frac = 0.1         # blank pixels below this fraction of peak smoothed coverage
off = ~nd_on

# pointing -> RA/Dec per dump
ant0 = scan_data.antennas[0]
loc = EarthLocation(lat=ant0.ref_observer.lat * u.rad, lon=ant0.ref_observer.lon * u.rad,
                    height=ant0.ref_observer.elevation * u.m)
az, el = calculate_median_coordinates_excluding_flagged_antennas(scan_data)
el = np.clip(el, 0., 90.)  # guard against near-zenith coordinate artefacts
times = Time(scan_data.timestamps.squeeze, format="unix")
radec = SkyCoord(az=az * u.deg, alt=el * u.deg,
                 frame=AltAz(obstime=times, location=loc)).icrs
ra, dec = radec.ra.deg, radec.dec.deg

# 2c residual at the channel nearest 700 MHz, flags masked
i_freq = int(np.argmin(np.abs(freq_MHz - freq_target_MHz)))
flags = np.asarray(scan_data.flags.combine(threshold=1).get(recv=i_recv).squeeze).astype(bool)
vis_off = np.ma.masked_array(vis[off, :, i_recv], mask=flags[off])
resid = vis_off - np.ma.median(vis_off, axis=0)[np.newaxis, :]
val = resid[:, i_freq]

good = ~np.ma.getmaskarray(val)
ra_g, dec_g, v_g = ra[off][good], dec[off][good], np.asarray(val[good])

# grid sum(value) and count, smooth both, divide (coverage-weighted mean)
ra_edges = np.arange(ra_g.min() - 1.0, ra_g.max() + 1.0 + pix_deg, pix_deg)
dec_edges = np.arange(dec_g.min() - 1.0, dec_g.max() + 1.0 + pix_deg, pix_deg)
sum_v = np.histogram2d(ra_g, dec_g, bins=[ra_edges, dec_edges], weights=v_g)[0]
cnt = np.histogram2d(ra_g, dec_g, bins=[ra_edges, dec_edges])[0]
sig = smoothing_deg / pix_deg
sm_v = ndi.gaussian_filter(sum_v, sig)
sm_c = ndi.gaussian_filter(cnt, sig)

smap = np.full_like(sm_v, np.nan)
m = sm_c > cover_frac * sm_c.max()
smap[m] = sm_v[m] / sm_c[m]

vlim = np.nanpercentile(np.abs(smap), 99)
cmap = plt.cm.RdBu_r.copy(); cmap.set_bad("white")
fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(smap.T, origin="lower", aspect="equal", cmap=cmap, vmin=-vlim, vmax=vlim,
               extent=[ra_edges[0], ra_edges[-1], dec_edges[0], dec_edges[-1]])
ax.set_xlabel("RA [deg]"); ax.set_ylabel("Dec [deg]")
ax.invert_xaxis()
ax.set_title(f"Smoothed ({smoothing_deg}\u00b0) sky map of 2c residual at {freq_MHz[i_freq]:.1f} MHz "
             f"\u2014 {receivers[i_recv]}, ND off")
plt.colorbar(im, label=r"$\Delta T$ [K]")
plt.tight_layout(); plt.show()

print(f"channel {i_freq} = {freq_MHz[i_freq]:.2f} MHz | mapped dumps: {good.sum()} | "
      f"map pixels filled: {int(m.sum())}")

## 2e-2. Same smoothed map, point-source positions overlaid

The 2e residual map is built from the simulated `vis`, which already contains the point sources, so
they are present in this map. Here the same smoothed map is re-plotted with the catalog source
positions marked (lime circles) so the simulated point sources can be identified as the positive
peaks at their sky positions. The print-out confirms the brightest in-strip sources sit well above
the foreground residual RMS.

In [ ]:
from museek.model import point_source_catalog as psc

extent = [ra_edges[0], ra_edges[-1], dec_edges[0], dec_edges[-1]]   # from cell 2e's grid
cat = psc.load_catalog("/home/mgrsantos/projects/museek/museek/model/1Jy_cat.txt", min_flux_Jy=1.0)
sel = psc.select_near_track(cat, ra[::8], dec[::8], 6.0)
cat = {k: v[sel] for k, v in cat.items()}
S700 = psc.flux_Jy(cat, np.array([700e6]))[:, 0]

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(smap.T, origin="lower", aspect="equal", cmap=cmap, vmin=-vlim, vmax=vlim, extent=extent)
ax.scatter(cat["ra_deg"], cat["dec_deg"], marker="o", s=90, facecolors="none",
           edgecolors="lime", linewidths=1.3, label="catalog sources")
ax.set_xlim(extent[1], extent[0]); ax.set_ylim(extent[2], extent[3])   # frame to the map, RA inverted
ax.set_xlabel("RA [deg]"); ax.set_ylabel("Dec [deg]")
ax.set_title(f"Smoothed residual at {freq_MHz[i_freq]:.0f} MHz with point-source positions (lime)")
plt.colorbar(im, label=r"$\Delta T$ [K]"); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

# quantify: smoothed-map value at the brightest in-strip sources vs the field RMS
ra_c = 0.5 * (ra_edges[:-1] + ra_edges[1:]); dec_c = 0.5 * (dec_edges[:-1] + dec_edges[1:])
field_rms = np.nanstd(smap)
strip = (np.abs(cat["dec_deg"]) < 6) & (cat["ra_deg"] > 132) & (cat["ra_deg"] < 163)
print(f"field residual RMS: {field_rms*1e3:.0f} mK")
for k in np.argsort(-S700 * strip)[:5]:
    if not strip[k]:
        continue
    ix = int(np.argmin(np.abs(ra_c - cat["ra_deg"][k]))); iy = int(np.argmin(np.abs(dec_c - cat["dec_deg"][k])))
    print(f"  {cat['source_id'][k]:>10} S700={S700[k]:4.1f} Jy: map peak {smap[ix, iy]*1e3:4.0f} mK "
          f"({smap[ix, iy]/field_rms:.1f}x RMS)")


## 2f. Comparison with the pysm3 synchrotron model

Same pysm3 `s1` model the plugin uses (700 MHz, equatorial), sampled at the scan pointings, median
removed, gridded/smoothed identically to 2e. Three panels: the **simulated** beam-convolved residual;
the model **point-sampled** (full HEALPix resolution); and the model **beam-smoothed** with the
actual primary beam (Gaussian-equivalent FWHM from the beam file's solid angle at 700 MHz).

The grid smoothing is identical for all three (shared `sig`). The point-sampled model shows extra
fine structure purely because it is **not** beam-convolved; once the beam is applied it should match
the simulation.

_(Reuses `ra_g, dec_g, ra_edges, dec_edges, sm_c, sig, m, smap, vlim, i_freq` from cell 2e.)_

In [ ]:
import astropy.units as u
import healpy as hp
import pysm3

BEAM_FILE = "/home/mgrsantos/projects/data/MeerKAT_U_band_primary_beam_aa_highres.npz"

# pysm3 s1 at the 700 MHz channel, rotated G->C (same as SimulateScanPlugin)
sky = pysm3.Sky(nside=128, preset_strings=["s1"])
emission = sky.get_emission(freq_MHz[i_freq] / 1e3 * u.GHz).value[0] / 1e6     # K_RJ (Galactic)
emission_eq = hp.Rotator(coord=["G", "C"]).rotate_map_pixel(emission)          # equatorial

# beam-equivalent Gaussian FWHM at this freq. NOTE: numpy's NpzFile ignores mmap_mode, so
# `np.load(...)["beam"]` would pull the full ~8 GB beam array into RAM. The beam npz is
# uncompressed, so we memory-map just the single frequency slice we need (a few MB) instead.
import struct
import zipfile

from numpy.lib import format as _npf


def _npz_member_memmap(npz_path, member):
    """Memmap one array from an *uncompressed* (STORED) .npz without loading the whole thing."""
    name = member if member.endswith(".npy") else member + ".npy"
    info = zipfile.ZipFile(npz_path).getinfo(name)
    if info.compress_type != zipfile.ZIP_STORED:
        raise ValueError(f"{name} is compressed; cannot memmap")
    with open(npz_path, "rb") as fh:
        fh.seek(info.header_offset + 26)
        n_name, n_extra = struct.unpack("<HH", fh.read(4))   # local-header name/extra lengths
        fh.seek(info.header_offset + 30 + n_name + n_extra)
        major, _minor = _npf.read_magic(fh)
        shape, fortran, dtype = (_npf.read_array_header_2_0(fh) if major >= 2
                                 else _npf.read_array_header_1_0(fh))
        data_off = fh.tell()
    return np.memmap(npz_path, dtype=dtype, mode="r", offset=data_off,
                     shape=shape, order="F" if fortran else "C")

with np.load(BEAM_FILE) as _bd:                 # reads only the small members, not 'beam'
    beam_freq = np.asarray(_bd["freq_MHz"]); margin = np.asarray(_bd["margin_deg"])
i_bf = int(np.argmin(np.abs(beam_freq - freq_MHz[i_freq])))
beam_mm = _npz_member_memmap(BEAM_FILE, "beam")                 # (4, 1, nfreq, nm, nl) memmap
power = np.abs(np.asarray(beam_mm[0, 0, i_bf])) ** 2            # (n_m, n_l) HH power, ~2 MB read
omega_b = power.sum() * np.deg2rad(margin[1] - margin[0]) ** 2                  # sr
beam_fwhm_rad = np.sqrt(omega_b / 1.133)                                        # Gaussian-equiv FWHM
beam_fwhm_deg = np.degrees(beam_fwhm_rad)
emission_beam = hp.smoothing(emission_eq, fwhm=beam_fwhm_rad)

def grid_resid(map_eq):
    vals = hp.get_interp_val(map_eq, np.radians(90.0 - dec_g), np.radians(ra_g % 360.0))
    resid = vals - np.median(vals)
    s = np.histogram2d(ra_g, dec_g, bins=[ra_edges, dec_edges], weights=resid)[0]
    out = np.full_like(s, np.nan)
    out[m] = ndi.gaussian_filter(s, sig)[m] / sm_c[m]
    return out

mmap_point = grid_resid(emission_eq)
mmap_beam = grid_resid(emission_beam)

cmap = plt.cm.RdBu_r.copy(); cmap.set_bad("white")
extent = [ra_edges[0], ra_edges[-1], dec_edges[0], dec_edges[-1]]
panels = [(smap, "Simulated (beam-convolved)"),
          (mmap_point, "pysm3 s1 (point-sampled)"),
          (mmap_beam, f"pysm3 s1 (beam-smoothed, {beam_fwhm_deg:.2f}\u00b0 FWHM)")]
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
for ax, (data, title) in zip(axes, panels):
    im = ax.imshow(data.T, origin="lower", aspect="equal", cmap=cmap, vmin=-vlim, vmax=vlim, extent=extent)
    ax.set_xlabel("RA [deg]"); ax.invert_xaxis(); ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, label=r"$\Delta T$ [K]")
axes[0].set_ylabel("Dec [deg]")
fig.suptitle(f"2c residual vs synchrotron model at {freq_MHz[i_freq]:.1f} MHz "
             f"(shared scale \u00b1{vlim:.2f} K, grid smoothing {smoothing_deg}\u00b0)")
plt.tight_layout(); plt.show()

print(f"beam-equiv FWHM at {freq_MHz[i_freq]:.0f} MHz: {beam_fwhm_deg:.2f} deg")
print(f"corr sim vs point-sampled model: {np.corrcoef(smap[m], mmap_point[m])[0,1]:.3f}")
print(f"corr sim vs beam-smoothed model: {np.corrcoef(smap[m], mmap_beam[m])[0,1]:.3f}")
print(f"contrast  point/sim = {np.nanstd(mmap_point)/np.nanstd(smap):.2f} | "
      f"beam/sim = {np.nanstd(mmap_beam)/np.nanstd(smap):.2f}")

## 2g. Point sources — method A vs method B

The two configurable point-source options computed independently for one receiver at 700 MHz:
**A** rasterises each catalog source into a HEALPix map and convolves it with Simeer (position
quantised to the nside-128 pixel); **B** evaluates the primary beam at each source's exact `(l,m)`
offset per dump. Catalog source positions are marked; both should light up the scan track at the
same places, and agree in amplitude up to method-A pixelisation.

Dumps flagged by `antenna_flagger` (e.g. the stowed/zenith excursion, where the pointing is invalid) are excluded — there is no point evaluating the sky where the data is already flagged.

In [ ]:
# §2g is a heavy DIAGNOSTIC: it loads the full ~8 GB beam (MeerKLASSBeam) to cross-check the
# two point-source injection methods. The actual simulation uses method B (primary_beam), so
# this comparison is OFF by default. Set RUN_METHOD_A = True to run it.
RUN_METHOD_A = False

if not RUN_METHOD_A:
    psA = psB = good = None
    print("§2g skipped (RUN_METHOD_A=False). Set it True to run the method-A vs B comparison "
          "(loads the full ~8 GB beam via MeerKLASSBeam).")
else:
    import astropy.units as u
    import healpy as hp
    from astropy.coordinates import AltAz, EarthLocation, SkyCoord
    from astropy.time import Time

    from museek.external.simeer import MeerKLASSBeam, integrate_tod
    from museek.model import point_source_catalog as psc
    from museek.model.primary_beam import PrimaryBeam
    from museek.plugin.point_source_calibration_plugin import (
        calculate_median_coordinates_excluding_flagged_antennas,
    )

    BEAM_FILE = "/home/mgrsantos/projects/data/MeerKAT_U_band_primary_beam_aa_highres.npz"
    CATALOG = "/home/mgrsantos/projects/museek/museek/model/1Jy_cat.txt"
    freq_target_MHz = 700.0
    pol = "HH"

    # pointing geometry (matches the simulation)
    ant0 = scan_data.antennas[0]
    loc = EarthLocation(lat=ant0.ref_observer.lat * u.rad, lon=ant0.ref_observer.lon * u.rad,
                        height=ant0.ref_observer.elevation * u.m)
    lat_deg = float(np.degrees(ant0.ref_observer.lat))
    az, el = calculate_median_coordinates_excluding_flagged_antennas(scan_data)
    el = np.clip(el, 0., 90.)  # guard against near-zenith coordinate artefacts
    times = Time(scan_data.timestamps.squeeze, format="unix", location=loc)
    lst = times.sidereal_time("apparent").deg
    radec = SkyCoord(az=az * u.deg, alt=el * u.deg, frame=AltAz(obstime=times, location=loc)).icrs
    ra, dec = radec.ra.deg, radec.dec.deg

    # catalog sources within 6 deg of the track (same selection the plugin uses)
    cat = psc.load_catalog(CATALOG, min_flux_Jy=1.0)
    sel = psc.select_near_track(cat, ra[::8], dec[::8], 6.0)
    cat = {k: v[sel] for k, v in cat.items()}

    beam = MeerKLASSBeam(BEAM_FILE, antenna="array_average", polarizations=("HH", "VV"))
    pb = PrimaryBeam(BEAM_FILE)
    fch = beam.freq_MHz[int(np.argmin(np.abs(beam.freq_MHz - freq_target_MHz)))]
    nu = fch * 1e6
    S = psc.flux_Jy(cat, np.array([nu]))[:, 0]

    # --- Method A: source-only HEALPix cube -> Simeer ---
    nside = 128
    omega_pix = 4.0 * np.pi / hp.nside2npix(nside)
    cube = np.zeros((1, hp.nside2npix(nside)))
    t_pix = psc.jy_to_kelvin(S, nu, omega_pix)
    pix = hp.ang2pix(nside, np.radians(90 - cat["dec_deg"]), np.radians(cat["ra_deg"] % 360))
    for k in range(len(pix)):
        cube[0, pix[k]] += t_pix[k]
    psA = integrate_tod(lst_deg_list=lst, az_deg_list=az, el_deg_list=el, lat_deg=lat_deg,
                        beam=beam, sky_maps=cube, freq_MHz=np.array([fch]),
                        disc_radius_deg=8.0, polarization=pol, n_jobs=6)[0]

    # --- Method B: PrimaryBeam analytic at exact source offsets ---
    ob = pb.get_beam_solid_angle_at_freq(np.array([fch]), pol)[0]
    t_peak = psc.jy_to_kelvin(S, nu, ob)
    psB = np.zeros(len(az))
    for k in range(len(cat["ra_deg"])):
        sc = SkyCoord(ra=cat["ra_deg"][k] * u.deg, dec=cat["dec_deg"][k] * u.deg)
        aa = sc.transform_to(AltAz(obstime=times, location=loc))
        near = sc.separation(radec).deg < 6.0
        if not near.any():
            continue
        g = pb.get_beam_gain(az[near], el[near], aa.az.deg[near], aa.alt.deg[near], np.array([fch]), pol)[:, 0]
        psB[near] += g * t_peak[k]

    print(f"{len(sel)} sources, channel {fch:.1f} MHz | method A max {psA.max()*1e3:.0f} mK, method B max {psB.max()*1e3:.0f} mK")


    # exclude dumps flagged by antenna_flagger (stowed/zenith excursion -> invalid pointing)
    fnames = scan_data.flags.flag_names
    def _dump_flag(name):
        return np.asarray(scan_data.flags.get(freq=0, recv=0).array[fnames.index(name)]).squeeze().astype(bool)
    ant_flagged = np.zeros(len(az), bool)
    for nm in ('elevation_flag', 'outlier_antenna_flag'):
        if nm in fnames:
            ant_flagged |= _dump_flag(nm)
    good = (~nd_on) & (~ant_flagged)
    print(f'valid (ND-off, not antenna-flagged) dumps: {int(good.sum())}/{len(good)} (antenna-flagged/stowed: {int(ant_flagged.sum())})')


    # free the large beam objects: they are not needed by any later cell
    import gc
    del beam, pb
    gc.collect()

In [ ]:
if not RUN_METHOD_A:
    print("§2g plot skipped (RUN_METHOD_A=False).")
else:
    off = good   # ND-off AND not antenna-flagged (excludes stowed dumps)
    vlim = np.percentile(np.concatenate([psA[off], psB[off]]) * 1e3, 99.7)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    for ax, ps, title in ((axes[0], psA, "Method A (HEALPix + Simeer)"),
                          (axes[1], psB, "Method B (PrimaryBeam)")):
        sc = ax.scatter(ra[off], dec[off], c=ps[off] * 1e3, s=9, cmap="inferno", vmin=0, vmax=vlim)
        ax.scatter(cat["ra_deg"], cat["dec_deg"], marker="x", s=70, c="cyan", linewidths=1.3, label="catalog")
        ax.set_xlim(ra[off].max() + 1, ra[off].min() - 1)   # RA increases to the left
        ax.set_ylim(dec[off].min() - 1.5, dec[off].max() + 1.5)
        ax.set_xlabel("RA [deg]"); ax.set_title(title); ax.legend(loc="upper right", fontsize=8)
        plt.colorbar(sc, ax=ax, label=r"$\Delta T$ [mK]", fraction=0.046)
    axes[0].set_ylabel("Dec [deg]")
    fig.suptitle(f"Point-source antenna temperature at {fch:.0f} MHz ({pol}, ND off) \u2014 catalog positions in cyan")
    plt.tight_layout(); plt.show()


In [ ]:
if not RUN_METHOD_A:
    print("§2g plot skipped (RUN_METHOD_A=False).")
else:
    off = good   # ND-off AND not antenna-flagged (excludes stowed dumps)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(psB[off] * 1e3, psA[off] * 1e3, s=4, alpha=0.3)
    lim = max(psA[off].max(), psB[off].max()) * 1e3
    ax.plot([0, lim], [0, lim], "k--", lw=1, label="1:1")
    ax.set_xlabel("Method B (PrimaryBeam) [mK]"); ax.set_ylabel("Method A (Simeer) [mK]")
    ax.set_title(f"Point-source TOD per dump: A vs B ({fch:.0f} MHz)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    bright = psB[off] * 1e3 > 20
    print(f"per-dump correlation A vs B: {np.corrcoef(psA[off], psB[off])[0,1]:.3f}")
    print(f"A/B ratio (dumps > 20 mK in B): median {np.median(psA[off][bright] / psB[off][bright]):.3f} "
          f"(method A < B by the nside-128 pixelisation)")


## 3. Spectra at a few times

Smooth synchrotron + system temperature; the dips/rises at the band edges are the
receiver/noise-diode model extrapolation (580–1015 MHz model vs 544–1088 MHz band).

In [ ]:
i_recv = 0
fig, ax = plt.subplots(figsize=(11, 5))
for t in [0, vis.shape[0] // 2, vis.shape[0] - 1]:
    tag = "ND on" if nd_on[t] else "ND off"
    ax.plot(freq_MHz, np.ma.filled(vism[t, :, i_recv], np.nan), lw=0.7, label=f"t={t_min[t]:.0f} min ({tag})")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel("$T_{\\rm ant}$ [K]")
ax.set_title(f"Simulated spectrum \u2014 {receivers[i_recv]} (flags masked)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Time series + noise-diode injection

Median over the clean mid-band vs time. The red points are `noise_diode_on` dumps and should
sit a clear step above the baseline; the baseline itself drifts with pointing/elevation.

In [ ]:
i_recv = 0
series = np.ma.filled(np.ma.median(vism[:, band, i_recv], axis=1), np.nan)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(t_min, series, lw=0.6, color="C0", label=receivers[i_recv])
ax.plot(t_min[nd_on], series[nd_on], ".", ms=5, color="red", label="noise diode on")
ax.set_xlabel("Time [min]")
ax.set_ylabel(r"median$_\nu$ $T_{\rm ant}$ [K]")
ax.set_title("Sky drift + noise-diode injection (flags masked)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4b. Median over frequency (diode OFF) vs time — all receivers

One curve per receiver: the median over all frequency channels at each `noise_diode_on == False`
dump. With the diode dumps removed this traces the foreground/system-temperature drift; the
per-receiver offsets are the different receiver temperatures.

In [ ]:
time_off = t_min[~nd_on]

fig, ax = plt.subplots(figsize=(13, 5))
for i, name in enumerate(receivers):
    med_off = np.ma.median(vism[~nd_on, :, i], axis=1)   # median over freq, ND-off dumps
    ax.plot(time_off, np.ma.filled(med_off, np.nan), lw=0.7, label=name)
ax.set_xlabel("Time [min]")
ax.set_ylabel(r"median$_\nu$ $T_{\rm ant}$ [K]  (ND off)")
ax.set_title("Median over frequency vs time (noise-diode OFF) \u2014 all receivers (flags masked)")
ax.legend(ncol=3, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4c. Median$_\nu$ $T_{\rm ant}$ (diode OFF) with elevation overlaid

Same median-over-frequency curves, with the pointing **elevation** (dashed, right axis) on top.
This makes clear that temperature excursions (e.g. the bump near ~95–100 min) track pointing
changes — here a slew to the **zenith** (el ≈ 90°) and back, rather than normal el ≈ 40° scanning.

In [ ]:
from museek.plugin.point_source_calibration_plugin import (
    calculate_median_coordinates_excluding_flagged_antennas,
)

_, el_deg = calculate_median_coordinates_excluding_flagged_antennas(scan_data)
el_deg = np.clip(el_deg, 0., 90.)
time_off = t_min[~nd_on]

fig, ax = plt.subplots(figsize=(13, 5))
for i, name in enumerate(receivers):
    med_off = np.ma.median(vism[~nd_on, :, i], axis=1)
    ax.plot(time_off, np.ma.filled(med_off, np.nan), lw=0.7, label=name)
ax.set_xlabel("Time [min]")
ax.set_ylabel(r"median$_\nu$ $T_{\rm ant}$ [K]  (ND off)")
ax.grid(alpha=0.3)

ax2 = ax.twinx()
ax2.plot(time_off, el_deg[~nd_on], "k--", lw=1.0, alpha=0.6)
ax2.set_ylabel("Elevation [deg]")

ax.set_title("Median$_\\nu$ $T_{\\rm ant}$ (ND off) with pointing elevation (dashed, flags masked)")
ax.legend(ncol=3, fontsize=9, loc="upper left")
plt.tight_layout()
plt.show()

## 4d. Per-dish elevation vs time (median sanity check)

Elevation of each individual dish with the median (used by the simulation) dashed on top,
**with flagged times masked** — the coordinated zenith excursion caught by `elevation_flag`
(plus any per-dish `outlier_antenna_flag`). What remains is the science scanning elevation;
the per-dump spread across dishes over the unflagged data is < 0.6°, so the dishes track
together and the median is a faithful summary of the pointing.

In [ ]:
from museek.plugin.point_source_calibration_plugin import (
    calculate_median_coordinates_excluding_flagged_antennas,
)

el_data = np.asarray(scan_data.elevation.squeeze)        # (n_time, n_antennas)
antennas = scan_data.antennas
antenna_names = [a.name for a in antennas]
_, el_med = calculate_median_coordinates_excluding_flagged_antennas(scan_data)

fnames = scan_data.flags.flag_names

def _antenna_time_flag(i_ant):
    """Per-dump flag for antenna `i_ant`: elevation_flag OR outlier_antenna_flag (any freq)."""
    i_recv = scan_data.receiver_indices_of_antenna(antennas[i_ant])[0]
    flagged = np.zeros(el_data.shape[0], dtype=bool)
    for nm in ("elevation_flag", "outlier_antenna_flag"):
        if nm in fnames:
            arr = np.asarray(scan_data.flags.get(recv=i_recv).array[fnames.index(nm)]).squeeze()
            flagged |= arr.astype(bool).any(axis=1)
    return flagged

# per-dump mask for the median: the coordinated excursion flagged by elevation_flag
med_flag = np.zeros(el_data.shape[0], dtype=bool)
if "elevation_flag" in fnames:
    med_flag = np.asarray(
        scan_data.flags.get(recv=0).array[fnames.index("elevation_flag")]
    ).squeeze().astype(bool).any(axis=1)

fig, ax = plt.subplots(figsize=(13, 5))
for i, name in enumerate(antenna_names):
    el_i = np.ma.masked_where(_antenna_time_flag(i), el_data[:, i])
    ax.plot(t_min, np.ma.filled(el_i, np.nan), lw=0.9, label=name)
ax.plot(t_min, np.ma.filled(np.ma.masked_where(med_flag, el_med), np.nan),
        "k--", lw=1.3, alpha=0.7, label="median (used by sim)")
ax.set_xlabel("Time [min]")
ax.set_ylabel("Elevation [deg]")
ax.set_title("Per-dish elevation vs time (flagged times masked)")
ax.legend(ncol=4, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

ok = ~med_flag   # unflagged dumps
print(f"flagged-time fraction: {med_flag.mean()*100:.1f}%")
print("max per-dump spread across dishes (unflagged):", f"{np.ptp(el_data[ok], axis=1).max():.3f} deg")

## 4e. Applying the flags — `elevation_flag` removes the zenith excursion

`antenna_flagger` **does** flag the zenith dumps (`elevation_flag` = 100% there, 0% on the normal
scan). The simulation writes a value for every dump regardless of flags, so the excursion only
shows because the earlier plots don't apply the flags. Here we mask `vis` with the combined flags
and take the median over *unflagged* channels — the ~95–100 min bump becomes a gap, confirming it
is correctly flagged.

In [ ]:
all_flags = scan_data.flags.combine(threshold=1)   # combined boolean flags (n_time, n_freq, n_recv)
time_off = t_min[~nd_on]

fig, ax = plt.subplots(figsize=(13, 5))
for i, name in enumerate(receivers):
    fl = np.asarray(all_flags.get(recv=i).squeeze).astype(bool)        # (n_time, n_freq)
    masked = np.ma.masked_array(vis[:, :, i], mask=fl)
    med = np.ma.median(masked, axis=1)                                 # masked where the whole dump is flagged
    med = np.ma.masked_where(nd_on, med)                               # also drop noise-diode dumps
    ax.plot(t_min, med, lw=0.7, label=name)
ax.set_xlabel("Time [min]")
ax.set_ylabel(r"median$_\nu$ $T_{\rm ant}$ [K]  (flagged channels & ND excluded)")
ax.set_title("Median over frequency vs time, flags applied \u2014 zenith excursion removed")
ax.legend(ncol=3, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

elev = np.asarray(scan_data.flags.get(recv=0).array[scan_data.flags.flag_names.index("elevation_flag")]).squeeze()
print("elevation_flag covers", f"{elev.astype(bool).any(axis=1).mean()*100:.1f}%", "of dumps")

## 5. Per-receiver comparison & noise-diode excess spectrum

Left: off-dump mean spectrum per receiver (receiver-temperature differences show here).
Right: the noise-diode excess spectrum `T(on) - T(off)` — should match duty×T_nd.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, name in enumerate(receivers):
    off_spec = vism[~nd_on][:, :, i].mean(axis=0) if nd_on.any() else vism[:, :, i].mean(axis=0)
    axes[0].plot(freq_MHz, np.ma.filled(off_spec, np.nan), lw=0.7, label=name)
axes[0].set_xlabel("Frequency [MHz]"); axes[0].set_ylabel(r"$T_{\rm sys}$ (ND off) [K]")
axes[0].set_title("Mean spectrum per receiver (ND off, flags masked)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

if nd_on.any():
    for i, name in enumerate(receivers):
        excess = vism[nd_on][:, :, i].mean(axis=0) - vism[~nd_on][:, :, i].mean(axis=0)
        axes[1].plot(freq_MHz, np.ma.filled(excess, np.nan), lw=0.7, label=name)
    axes[1].set_xlabel("Frequency [MHz]"); axes[1].set_ylabel(r"$T_{\rm ant}$(on) - (off) [K]")
    axes[1].set_title("Noise-diode excess spectrum (flags masked)"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 6. Synthetic-gain explorer

Dial in the synthetic gain (`gain_smooth_poly` x `(1 + standing waves)`) and see it across the band,
using `SimulateScanPlugin._synthetic_gain` (the exact formula the sim applies). Standing waves are
`(displacement_m, amplitude, phase)`: displacement sets the ripple period `Δf = 150/d` MHz, amplitude
its fractional depth, phase slides it. `gain_smooth_poly=None` (or `[1.0]`) -> flat smooth = 1; set
e.g. `[10.0, 0.5]` for a 10 counts/K level with a slope. This cell is self-contained (no data needed).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from museek.plugin.simulate_scan_plugin import SimulateScanPlugin

# ----------------- dial in the synthetic gain here -----------------
gain_smooth_poly = None                         # None or [1.0] -> flat 1; e.g. [10.0, 0.5] for level+slope
gain_standing_waves = [(1.0, 0.03, 0.0),        # (displacement_m, amplitude, phase)
                       (15.0, 0.01, 1.5)]
freq_grid = np.linspace(544.0, 1088.0, 4096)    # MeerKAT U band
# -------------------------------------------------------------------

_sim = SimulateScanPlugin(beam_file_path="", receiver_models_dir="", noise_diodes_dir="",
                          spillover_model_file="", gain_smooth_poly=gain_smooth_poly,
                          gain_standing_waves=gain_standing_waves)
gain = _sim._synthetic_gain(freq_grid)
if gain is None:
    gain = np.ones_like(freq_grid)

print("standing waves:")
for d, a, p in (gain_standing_waves or []):
    print(f"  displacement {d:>5} m  ->  period {150.0/d:6.1f} MHz,  amplitude {a},  phase {p}")
print(f"gain range over band: [{gain.min():.3f}, {gain.max():.3f}]")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(freq_grid, gain, lw=0.8)
ax.set_xlabel("Frequency [MHz]"); ax.set_ylabel("synthetic gain factor")
ax.set_title("Synthetic gain = smooth_poly(f) x (1 + standing waves)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 7. HI check (Gaussian-field mock)

Self-contained check of the limTOD HI field the sim injects. Shows a sky map, the **frequency–
frequency correlation** (near-diagonal = decorrelated across frequency, the signature that
distinguishes HI from the smooth foreground), and a few spectra (the spectral roughness). Edit the
`xi` (frequency correlation length) to see the decorrelation tighten/loosen.

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

from museek.external.limtod import generate_gaussian_field

# --- HI field parameters (same model the sim uses) ---
nside = 64
freqs = np.linspace(580.0, 1015.0, 128)             # MHz
xi = 0.01                                           # frequency correlation length (small -> decorrelated)
target_rms_mK = 0.3

cube = generate_gaussian_field(freqs=freqs, nside=nside, amp=1.0, alpha=-1.0, beta=1.0, xi=xi, seed=0)
cube = cube * (target_rms_mK * 1e-3 / cube.std())   # (n_freq, n_pix) in K

i700 = int(np.argmin(np.abs(freqs - 700.0)))
hp.mollview(cube[i700] * 1e3, title=f"HI map @ {freqs[i700]:.0f} MHz", unit="mK", cmap="RdBu_r")
hp.graticule()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
corr = np.corrcoef(cube)                            # (n_freq, n_freq) over pixels
im = axes[0].imshow(corr, origin="upper", cmap="RdBu_r", vmin=-1, vmax=1,
                    extent=[freqs[0], freqs[-1], freqs[-1], freqs[0]])
axes[0].set_xlabel("Frequency [MHz]"); axes[0].set_ylabel("Frequency [MHz]")
axes[0].set_title(f"HI freq-freq correlation (xi={xi}) -- near-diagonal = decorrelated")
plt.colorbar(im, ax=axes[0], fraction=0.046)
for p in (100, 2000, 30000):
    axes[1].plot(freqs, cube[:, p] * 1e3, lw=0.7)
axes[1].set_xlabel("Frequency [MHz]"); axes[1].set_ylabel("HI [mK]")
axes[1].set_title("HI spectra at a few pixels (spectrally rough)"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# decorrelation length: half-width where the correlation of the mid channel drops below 0.5
mid = len(freqs) // 2
below = np.where(corr[mid] < 0.5)[0]
if len(below):
    dnu = abs(freqs[below[np.argmin(np.abs(below - mid))]] - freqs[mid])
    print(f"freq correlation drops below 0.5 within ~{dnu:.1f} MHz of a channel (xi={xi})")
print(f"HI cube RMS: {cube.std()*1e3:.3f} mK")


## 8. Noise check (1/f gain + radiometer white)

Self-contained check of the two noise terms the sim applies. **1/f gain**: the fractional gain
fluctuation time series and its power spectrum (1/f rise above the white floor). **White
radiometer**: the per-sample scatter `1/sqrt(dnu*tau)` and its integrate-down (RMS ∝ 1/sqrt(N) as
you bin in time) -- which is how the faint HI is recovered from under the noise.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.fft import rfft, rfftfreq

from museek.external.limtod import sim_noise

np.random.seed(0)
n_time = 2975
dt = 2.0                                    # dump period [s]
t = np.arange(n_time) * dt                  # ~100 min
T_sys = 12.0                                # K, for converting fractional -> mK

# 1/f gain fluctuation (same call as the plugin)
dg = sim_noise(time_list=t, n_samples=1, f0=1.335e-5, fc=1.099e-3, alpha=2, white_n_variance=5e-6)[0]
# white radiometer fractional noise
dnu = 0.133e6                               # channel width [Hz]
radiometer = 1.0 / np.sqrt(dnu * dt)
white = radiometer * np.random.standard_normal(n_time)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(t / 60, dg * 100, lw=0.5)
axes[0, 0].set_xlabel("Time [min]"); axes[0, 0].set_ylabel("gain fluctuation [%]")
axes[0, 0].set_title(f"1/f gain fluctuation (RMS {dg.std()*100:.3f}%)"); axes[0, 0].grid(alpha=0.3)

psd = np.abs(rfft(dg - dg.mean())) ** 2; fr = rfftfreq(n_time, d=dt)
axes[0, 1].loglog(fr[1:], psd[1:], lw=0.6)
axes[0, 1].set_xlabel("Frequency [Hz]"); axes[0, 1].set_ylabel("PSD")
axes[0, 1].set_title("1/f gain PSD (rise toward low freq)"); axes[0, 1].grid(alpha=0.3, which="both")

axes[1, 0].hist(white * T_sys * 1e3, bins=60, color="0.5")
axes[1, 0].set_xlabel("white noise per sample [mK on 12 K]"); axes[1, 0].set_ylabel("count")
axes[1, 0].set_title(f"radiometer white noise: {radiometer*T_sys*1e3:.0f} mK/sample")

ns = np.array([1, 2, 5, 10, 20, 50, 100, 200])
rms = np.array([white[:n_time // n * n].reshape(-1, n).mean(1).std() for n in ns])
axes[1, 1].loglog(ns, rms / rms[0], "o-", label="measured")
axes[1, 1].loglog(ns, 1 / np.sqrt(ns), "k--", label=r"$1/\sqrt{N}$")
axes[1, 1].axhline((0.3e-3 / (T_sys * radiometer)), color="C3", ls=":", label="HI level (0.3 mK)")
axes[1, 1].set_xlabel("samples binned"); axes[1, 1].set_ylabel("relative RMS")
axes[1, 1].set_title("white noise integrates down"); axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()
print(f"1/f gain RMS {dg.std()*100:.3f}% | white {radiometer*T_sys*1e3:.0f} mK/sample ({radiometer*100:.3f}%)")
